# Ensembl Gene ID to Gene Name Converter

This notebook takes a list of Ensembl gene IDs like `ENSG00000174111` and tries to find their gene symbols/names.

It uses **MyGene.info** first, then shows which IDs were not mapped.

You can replace the list of IDs with your own Excel column later.

In [ ]:
# Install mygene if needed
!pip -q install mygene openpyxl pandas

In [ ]:
import pandas as pd
import mygene

# Your Ensembl gene IDs
gene_ids = [
    "ENSG00000174111",
    "ENSG00000238683",
    "ENSG00000130723",
    "ENSG00000254184",
    "ENSG00000255322",
    "ENSG00000262370",
    "ENSG00000233864",
    "ENSG00000205664",
    "ENSG00000241860",
    "ENSG00000260661",
    "ENSG00000272373",
    "ENSG00000243491",
]

gene_ids

## Query MyGene.info

This checks Ensembl IDs against human gene annotation.

In [ ]:
mg = mygene.MyGeneInfo()

results = mg.querymany(
    gene_ids,
    scopes="ensembl.gene",
    fields="symbol,name,type_of_gene,entrezgene,ensembl.gene",
    species="human",
    as_dataframe=False,
    returnall=False,
    verbose=False
)

results[:3]

## Clean the results into a table

In [ ]:
rows = []

for item in results:
    query_id = item.get("query")
    not_found = item.get("notfound", False)

    rows.append({
        "ensembl_gene_id": query_id,
        "found": not not_found,
        "gene_symbol": item.get("symbol"),
        "gene_name": item.get("name"),
        "gene_type": item.get("type_of_gene"),
        "entrez_id": item.get("entrezgene"),
        "mygene_id": item.get("_id"),
    })

df = pd.DataFrame(rows)
df

## Show only IDs that were not found

In [ ]:
not_found_df = df[df["found"] == False]
not_found_df

## Save the result as Excel and CSV

In [ ]:
df.to_excel("ensembl_gene_id_mapping.xlsx", index=False)
df.to_csv("ensembl_gene_id_mapping.csv", index=False)

print("Saved files:")
print("- ensembl_gene_id_mapping.xlsx")
print("- ensembl_gene_id_mapping.csv")

## Optional: load IDs from an Excel file

Use this section when your gene IDs are already in Excel.

Change:
- `your_file.xlsx`
- `Sheet1`
- `A`

to match your file, sheet, and column.

In [ ]:
# Example only. Uncomment and edit when needed.

# excel_file = "your_file.xlsx"
# sheet_name = "Sheet1"
# column_name = "A"  # If your file has headers, use the actual header name instead

# input_df = pd.read_excel(excel_file, sheet_name=sheet_name)
# gene_ids = input_df[column_name].dropna().astype(str).str.strip().tolist()
# gene_ids[:10]